# Previsão do Diagnóstico de Tumores Cerebrais
### Aprendizagem Automática - Licenciatura em Engenharia Informática

**Trabalho realizado por:**
- Francisco Rodrigues nº l59119


## Introdução
Este notebook documenta o trabalho desenvolvido para o desafio Kaggle "Diagnóstico de Tumores Cerebrais". O conjunto de dados combina atributos demográficos dos pacientes com medidas de textura extraídas de imagens de ressonância magnética (ADC e GLCM), sendo a unidade de predição o paciente (agregando múltiplas fatias por id).

O objetivo deste trabalho é construir modelos que classifiquem tumores como maligno (1) ou benigno (0), maximizar a métrica F1-Score e produzir submissões robustas para o Kaggle.

## Imports usados
Em baixo serão indicados os imports usados no código desenvolvido para a construção e submissão dos diferentes modelos testados:

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline


Além disso, também fiz import do(s) algoritmo(s) a usar, como por exemplo:

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Estes algoritmos serão usados para a demonstração do código mais à frente

## 2. Caracterização e Análise do Conjunto de Dados
Nesta secção, procedi ao carregamento dos dados fornecidos e a uma breve análise exploratória. É crucial verificar as dimensões iniciais e identificar o número de pacientes únicos, uma vez que o dataset original contém múltiplas fatias de ressonância magnética por paciente. Esta observação inicial é o que motiva a necessidade de agregar os dados, pois a a predição final terá de ser feita à escala do paciente e não da fatia.

In [3]:
# 1. Carregar os dados
treino_df = pd.read_csv('/kaggle/input/datasets/franciscouevora/datasetespecial/treino.csv')
teste_df = pd.read_csv('/kaggle/input/datasets/franciscouevora/datasetespecial/teste.csv')

# 2. Mostrar as primeiras linhas do conjunto de treino
display(treino_df.head())

# 3. Ver a dimensão inicial dos dados
print(f"Dimensão do treino: {treino_df.shape[0]} fatias e {treino_df.shape[1]} colunas.")
print(f"Dimensão do teste: {teste_df.shape[0]} fatias e {teste_df.shape[1]} colunas.")

# 4. Verificar quantos pacientes únicos existem no treino vs total de fatias
pacientes_unicos = treino_df['id'].nunique()
print(f"\nNo conjunto de treino existem {treino_df.shape[0]} fatias, mas pertencem a apenas {pacientes_unicos} pacientes únicos.")

,id,idade,sexo,id_fatia,med_ADC,assimetria,curtose,med1_GLCM,med2_GLCM,var1_GLCM,var2_GLCM,energia_GLCM,entropia_GLCM,contraste_GLCM,homogeneidade_GLCM,correlação_GLCM,proiminência_GLCM,sombra_GLCM,diagnostico
0,1,78,m,5.1,1208.988,11.092,2.774,9.817,9.860,14.637,16.047,0.052,3.507,14.646,0.462,0.523,1.427,1.855,1
1,1,78,m,5.2,897.146,3.044,0.169,7.171,7.482,1.068,2.981,0.081,2.887,3.262,0.571,0.248,1.576,2.076,1
2,1,78,m,5.3,895.563,7.827,2.068,7.133,7.457,2.850,6.443,0.076,3.135,5.941,0.550,0.403,1.809,2.405,1
3,1,78,m,5.4.1,1080.107,3.091,0.832,8.725,9.437,3.680,12.681,0.041,3.728,9.834,0.504,0.515,1.948,2.573,1
4,1,78,m,5.4.2,1025.789,14.450,3.005,8.282,8.736,4.202,12.335,0.067,3.204,9.046,0.548,0.535,2.158,2.920,1


Dimensão do treino: 1393 fatias e 19 colunas.
Dimensão do teste: 400 fatias e 18 colunas.

No conjunto de treino existem 1393 fatias, mas pertencem a apenas 186 pacientes únicos.


## 3. Pré-processamento e Agregação de Dados

Como o nosso conjunto de dados original está dividido por fatias de imagem por paciente (múltiplas linhas por ID), o primeiro passo para evitar a fuga de informação (*data leakage*) e cumprir os requisitos da avaliação é agregar a informação por paciente.
Para consolidar a informação e evitar fugas de dados (*data leakage*), criei a função `agregar_por_paciente`. Esta função agrupa todas as fatias pertencentes ao mesmo paciente e extrai estatísticas descritivas das variáveis numéricas de textura (média, desvio padrão, mínimo e máximo). Para a variável categórica do sexo, extraímos a moda (o valor mais frequente). O resultado é um conjunto de dados transformado, contendo exatamente uma linha independente por paciente.

In [4]:
# Função para agregar as fatias de imagem num vetor estatístico por paciente
def agregar_por_paciente(df, is_train=True):
    df_copy = df.copy()
    
    # Selecionar colunas numéricas de textura
    cols_numericas = [col for col in df_copy.columns if col not in ['id', 'id_fatia', 'sexo', 'diagnostico']]
    for col in cols_numericas:
        df_copy[col] = pd.to_numeric(df_copy[col], errors='coerce')
        
    # Calcular estatísticas descritivas para resumir o volume do tumor por paciente
    agg_funcs = {col: ['mean', 'std', 'min', 'max'] for col in cols_numericas}
    
    # Agrupar por ID do paciente
    df_num_agg = df_copy.groupby('id').agg(agg_funcs)
    df_num_agg.columns = [f"{col[0]}_{col[1]}" for col in df_num_agg.columns]
    df_num_agg = df_num_agg.reset_index()
    
    # Atributo categórico 'sexo': extrair o valor mais frequente (moda) por paciente
    df_cat_agg = df_copy.groupby('id')['sexo'].agg(lambda x: x.mode()[0] if not x.mode().empty else 'm').reset_index()
    
    # Combinar as características numéricas e categóricas
    df_final = pd.merge(df_num_agg, df_cat_agg, on='id')
    
    # Se for o conjunto de treino, associar o diagnóstico correto do paciente
    if is_train:
        df_diag = df_copy.groupby('id')['diagnostico'].max().reset_index()
        df_final = pd.merge(df_final, df_diag, on='id')
        
    return df_final

# Aplicar a agregação aos dados carregados anteriormente (treino_df e teste_df)
train_agg = agregar_por_paciente(treino_df, is_train=True)
test_agg = agregar_por_paciente(teste_df, is_train=False)

print(f"Agregação concluída com sucesso!")
print(f"Treino Agregado: {train_agg.shape[0]} pacientes e {train_agg.shape[1]} colunas.")
print(f"Teste Agregado: {test_agg.shape[0]} pacientes e {test_agg.shape[1]} colunas.")

Agregação concluída com sucesso!
Treino Agregado: 186 pacientes e 63 colunas.
Teste Agregado: 64 pacientes e 62 colunas.


## 4. Configuração dos Pipelines e Validação Cruzada
Antes de treinar os modelos, isolamos a matriz de características (`X`) da variável alvo (`y`). De seguida, configuramos as etapas de pré-processamento dentro de um `ColumnTransformer`:
* **Variáveis Numéricas:** Aplicação de `SimpleImputer` (para preencher eventuais valores omissos com a mediana) e `StandardScaler` (para normalizar a escala dos dados).
* **Variáveis Categóricas:** Aplicação de `OneHotEncoder` para binarizar o sexo do paciente.

Por fim, definimos a nossa estratégia de avaliação: validação cruzada estratificada com 5 partições (`StratifiedKFold`). Isto garante que a proporção de tumores malignos e benignos se mantém equilibrada em todas as dobras de treino e teste.

In [5]:
# 1. Separar a matriz de atributos (X) e a variável alvo (y) no conjunto de treino
X = train_agg.drop(columns=['id', 'diagnostico'])
y = train_agg['diagnostico']

# Guardar as características do conjunto de teste (excluindo o ID) para futuras previsões
X_test = test_agg.drop(columns=['id'])

# 2. Identificar os tipos de colunas presentes nas características
cols_numericas = X.select_dtypes(include=['float64', 'int64']).columns.tolist()
cols_categoricas = ['sexo']

# 3. Construção do sub-pipeline para os dados numéricos
# Inclui imputação (substituição de valores omissos pela mediana) e estandardização
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# 4. Construção do pré-processador completo utilizando o ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, cols_numericas),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cols_categoricas)
    ])

# 5. Configuração da Validação Cruzada Estratificada
# Garante que a proporção de classes (benigno/maligno) é mantida em cada fold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Pré-processador e Validação Cruzada configurados")
print(f"Atributos numéricos processados: {len(cols_numericas)}")
print(f"Atributos categóricos processados: {len(cols_categoricas)}")

Pré-processador e Validação Cruzada configurados
Atributos numéricos processados: 60
Atributos categóricos processados: 1


## Regressão Logística com Regularização Otimizada (C)
A Regressão Logística que testámos no início era muito básica. Se ajustarmos o parâmetro de regularização C, podemos controlar o quão estrito o modelo é, muitas vezes superando modelos complexos.

In [6]:
# Pipeline com Regressão Logística regularizada
modelo_lr = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(
        C=0.01,                          # Regularização mais forte
        penalty='l2', 
        class_weight='balanced',
        solver='liblinear',              # Solver para DataSets mais pequenos
        max_iter=1000,
        random_state=42
    ))
])

# Avaliar na validação cruzada
scores_lr = cross_val_score(modelo_lr, X, y, cv=cv, scoring='f1_weighted')

print("Resultados da Validação Cruzada (Regressão Logística Otimizada):")
print(f"F1-Scores em cada fold: {np.round(scores_lr, 4)}")
print(f"F1-Score Médio: {np.mean(scores_lr):.4f} (+/- {np.std(scores_lr):.4f})")
modelo_lr.fit(X, y)

Resultados da Validação Cruzada (Regressão Logística Otimizada):
F1-Scores em cada fold: [0.6254 0.779  0.8381 0.7535 0.6414]
F1-Score Médio: 0.7275 (+/- 0.0817)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['idade_mean', 'idade_std',
                                                   'idade_min', 'idade_max',
                                                   'med_ADC_mean',
                                                   'med_ADC_std', 'med_ADC_min',
                                                   'med_ADC_max',
                                                   'assimetria_mean',
                                                   'assimetria_std',
                                                   'assimetria_min',
                                                   'assimetria_max',
                                                   'curtose_mea...
                                                   'med2_GLCM_mean',
                                                   'med2_GLCM_std',
                                                   'med2_GLCM_min',
                                                   'med2_GLCM_max',
                                                   'var1_GLCM_mean',
                                                   'var1_GLCM_std',
                                                   'var1_GLCM_min',
                                                   'var1_GLCM_max',
                                                   'var2_GLCM_mean',
                                                   'var2_GLCM_std', ...]),
                                                 ('cat',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
                                                  ['sexo'])])),
                ('classifier',
                 LogisticRegression(C=0.01, class_weight='balanced',
                                    max_iter=1000, random_state=42,
                                    solver='liblinear'))])

## Random Forest com Restrição de Profundidade
Para este modelo, optei por um ensemble de árvores de decisão. Sendo algoritmos propensos a decorar os dados de treino (*overfitting*) – especialmente em datasets com poucos pacientes – apliquei restrições rigorosas. Limitamos a profundidade máxima das árvores (`max_depth=3`) e aumentamos o número de estimadores (`n_estimators=300`), forçando o modelo a focar-se apenas nos padrões mais fortes e gerais.

In [11]:
# 1. Criar o Pipeline com o Random Forest 
modelo_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=300, 
        max_depth=3,                        # Max_depth para 3
        min_samples_split=5, 
        random_state=42, 
        class_weight='balanced'
    ))
])

# 2. Avaliar com a mesma Validação Cruzada Estratificada
scores_rf = cross_val_score(modelo_rf, X, y, cv=cv, scoring='f1_weighted')

# 3. Mostrar os resultados
print("Resultados da Validação Cruzada (Random Forest):")
print(f"F1-Scores em cada fold: {np.round(scores_rf, 4)}")
print(f"F1-Score Médio: {np.mean(scores_rf):.4f} (+/- {np.std(scores_rf):.4f})")

# 4. Treinar em todos os dados
modelo_rf.fit(X, y)

Resultados da Validação Cruzada (Random Forest):
F1-Scores em cada fold: [0.6543 0.8919 0.7841 0.7822 0.8381]
F1-Score Médio: 0.7901 (+/- 0.0790)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['idade_mean', 'idade_std',
                                                   'idade_min', 'idade_max',
                                                   'med_ADC_mean',
                                                   'med_ADC_std', 'med_ADC_min',
                                                   'med_ADC_max',
                                                   'assimetria_mean',
                                                   'assimetria_std',
                                                   'assimetria_min',
                                                   'assimetria_max',
                                                   'curtose_mea...
                                                   'med2_GLCM_std',
                                                   'med2_GLCM_min',
                                                   'med2_GLCM_max',
                                                   'var1_GLCM_mean',
                                                   'var1_GLCM_std',
                                                   'var1_GLCM_min',
                                                   'var1_GLCM_max',
                                                   'var2_GLCM_mean',
                                                   'var2_GLCM_std', ...]),
                                                 ('cat',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
                                                  ['sexo'])])),
                ('classifier',
                 RandomForestClassifier(class_weight='balanced', max_depth=3,
                                        min_samples_split=5, n_estimators=300,
                                        random_state=42))])

## Máquinas de Vetores de Suporte (SVM - Support Vector Machines)
Os SVMs são excelentes para datasets pequenos ou de dimensões moderadas, pois encontram o hiperplano que melhor separa as classes maximizando a margem, o que reduz drasticamente o risco de overfitting.

In [8]:
# Pipeline com SVM
modelo_svm = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', SVC(
        kernel='linear', 
        C=0.01,
        probability=True, 
        class_weight='balanced', 
        random_state=42
    ))
])

# Avaliar na validação cruzada
scores_svm = cross_val_score(modelo_svm, X, y, cv=cv, scoring='f1_weighted')

print("Resultados da Validação Cruzada (SVM):")
print(f"F1-Scores em cada fold: {np.round(scores_svm, 4)}")
print(f"F1-Score Médio: {np.mean(scores_svm):.4f} (+/- {np.std(scores_svm):.4f})")

modelo_svm.fit(X, y)

Resultados da Validação Cruzada (SVM):
F1-Scores em cada fold: [0.5955 0.8643 0.8649 0.756  0.6016]
F1-Score Médio: 0.7365 (+/- 0.1194)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['idade_mean', 'idade_std',
                                                   'idade_min', 'idade_max',
                                                   'med_ADC_mean',
                                                   'med_ADC_std', 'med_ADC_min',
                                                   'med_ADC_max',
                                                   'assimetria_mean',
                                                   'assimetria_std',
                                                   'assimetria_min',
                                                   'assimetria_max',
                                                   'curtose_mea...
                                                   'med1_GLCM_max',
                                                   'med2_GLCM_mean',
                                                   'med2_GLCM_std',
                                                   'med2_GLCM_min',
                                                   'med2_GLCM_max',
                                                   'var1_GLCM_mean',
                                                   'var1_GLCM_std',
                                                   'var1_GLCM_min',
                                                   'var1_GLCM_max',
                                                   'var2_GLCM_mean',
                                                   'var2_GLCM_std', ...]),
                                                 ('cat',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
                                                  ['sexo'])])),
                ('classifier',
                 SVC(C=0.01, class_weight='balanced', kernel='linear',
                     probability=True, random_state=42))])

## 5. Geração das Submissões
Após a definição, treino e validação dos modelos, o passo final consiste em prever os diagnósticos sobre o conjunto de dados de teste (já devidamente agregado por paciente). O código abaixo itera sobre os modelos otimizados, gera as previsões finais e formata automaticamente os ficheiros CSV com as colunas `id` e `diagnostico` para submissão na plataforma Kaggle.

In [9]:
modelos_para_submeter = {
    'submissao_regressao_logistica.csv': modelo_lr,
    'submissao_random_forest.csv': modelo_rf,
    'submissao_svm.csv': modelo_svm
}

for nome_ficheiro, modelo in modelos_para_submeter.items():
    # 1. Gerar previsões no conjunto de teste
    previsoes_finais = modelo.predict(X_test)
    
    # 2. Criar a estrutura exigida pela plataforma Kaggle
    submissao_final = pd.DataFrame({
        'id': test_agg['id'],
        'diagnostico': previsoes_finais
    })
    
    # 3. Guardar num ficheiro CSV
    submissao_final.to_csv(nome_ficheiro, index=False)
    
    print(f"Ficheiro de submissão '{nome_ficheiro}'")
    display(submissao_final.head())
    print("-" * 100)

Ficheiro de submissão 'submissao_random_forest.csv'


,id,diagnostico
0,187,1
1,188,0
2,189,1
3,190,0
4,191,0


----------------------------------------------------------------------------------------------------
Ficheiro de submissão 'submissao_svm.csv'


,id,diagnostico
0,187,1
1,188,0
2,189,1
3,190,0
4,191,0


----------------------------------------------------------------------------------------------------
Ficheiro de submissão 'submissao_regressao_logistica.csv'


,id,diagnostico
0,187,1
1,188,0
2,189,1
3,190,0
4,191,0


----------------------------------------------------------------------------------------------------


## 6. Estratégia para Pesquisa e Escolha dos Modelos
A estratégia usada para a pesquisa e seleção de modelos baseou-se na experimentação de diferentes algoritmos clássicos abordados nas aulas teóricas. Para garantir a fiabilidade dos resultados e evitar o sobre-ajustamento (*overfitting*) a um conjunto de dados de dimensões reduzidas, utilizei sistematicamente a validação cruzada estratificada com 5 folds (`StratifiedKFold`) e a otimização através da métrica oficial do desafio (`f1_weighted`). 

Todos os algoritmos foram integrados em pipelines de pré-processamento para garantir que a imputação de dados omissos e a estandardização eram aplicadas de forma isolada em cada fold, prevenindo qualquer tipo de fuga de informação (*data leakage*).

## 7. Caracterização dos Modelos Submetidos
Durante o desenvolvimento, testei e comparei vários estimadores. Os três modelos finalistas selecionados para submissão foram:

### 7.1. Regressão Logística (Duas Variantes)
Para avaliar o impacto da força da regularização no desempenho final, testei diferentes configurações deste algoritmo:
* **Abordagem Inicial:** Modelo linear regularizado com penalização `l2`, parâmetro `C=0.1` e ajuste de pesos de classe (`class_weight='balanced'`).
  * **Desempenho (CV F1-Score Médio):** ~0.7210
* **Abordagem Final:** Aumento drástico da força de regularização (`C=0.01`) e alteração do solver paramétrico para `liblinear` (ideal para *datasets* pequenos).
  * **Desempenho (CV F1-Score Médio):** ~0.7275
* **Justificação:** A primeira abordagem já controlava a variabilidade dos coeficientes, mas a segunda variante demonstrou que um modelo ainda mais "duro" (com um `C` menor) restringe o *overfitting* de forma mais eficaz no conjunto de dados de treino reduzido, subindo a métrica de estabilidade.

### 7.2. Random Forest (Duas Variantes)
À semelhança da Regressão Logística, utilizei as submissões para testar o impacto da complexidade do modelo na sua capacidade de generalização:
* **Abordagem Inicial:** Ensemble configurado com uma profundidade máxima moderada (`max_depth=5`), `n_estimators=300` e `min_samples_split=5`.
  * **Observação:** Verificámos que, com uma profundidade de 5, o modelo tinha tendência para memorizar os dados de treino (*overfitting*), o que o tornava menos fiável em dados novos.
* **Abordagem Final:** Restrição da profundidade das árvores passando para `max_depth=3`, mantendo os restantes parâmetros e a ponderação de classes (`class_weight='balanced'`).
  * **Desempenho (CV F1-Score Médio):** ~0.7901
* **Justificação:** Em conjuntos de dados com um número reduzido de pacientes, árvores mais profundas tendem a criar regras demasiado específicas baseadas em ruído. A limitação drástica da profundidade para 3 foi crucial para "forçar" o modelo a focar-se apenas nos padrões globais mais fortes, melhorando significativamente a sua robustez preditiva.

### 7.3. Support Vector Machine - SVM (Duas Variantes)
O desempenho do SVM depende fortemente da complexidade geométrica da sua fronteira de decisão. Devido à alta dimensão do espaço de características, testámos duas abordagens distintas para evitar o sobre-ajustamento:
* **Abordagem Inicial:** Classificador com kernel não-linear RBF, utilizando regularização padrão (`C=1.0`) e pesos de classe balanceados (`class_weight='balanced'`).
  * **Desempenho (CV F1-Score Médio):** ~0.7279
* **Abordagem Final:** Transição para um **kernel linear** combinado com uma restrição severa através do aumento da força de regularização (`C=0.01`).
  * **Desempenho (CV F1-Score Médio):** ~0.7365
* **Justificação:** Embora o kernel RBF permita desenhar fronteiras curvas e complexas, num *dataset* com muitos atributos e poucos exemplos ele tem tendência para memorizar o ruído. A adoção de um kernel linear restrito por um `C` muito baixo demonstrou que um corte simples é mais do que suficiente para separar as classes, garantindo uma robustez e estabilidade muito superiores.

### 7.4. Resumo de Resultados Finais (Kaggle Leaderboard)
Com o término da competição, foi possível avaliar os modelos através da *Private Leaderboard* (que utiliza a totalidade dos dados de teste ocultos). A tabela abaixo cruza as estimativas locais (Validação Cruzada) com os resultados da plataforma para as diferentes variantes testadas:

| Modelo (Variante) | F1-Score Médio (CV) | F1-Score (Public LB) | F1-Score (Private LB) |
| :--- | :--- | :--- | :--- |
| **Regressão Logística (Inicial)** | ~0.7210 | 0.583 | 0.687 |
| **Regressão Logística (Otimizada C=0.01)** | ~0.7275 | 0.720 | 0.727 |
| **Random Forest (Inicial)** | ~0.7843 | 0.636 | 0.750 |
| **Random Forest (Otimizado max_depth=3)** | ~0.7901 | 0.695 | 0.750 |
| **SVM (Inicial / RBF)** | ~0.7279 | 0.769 | **0.764** |
| **SVM (Otimizado Linear C=0.01)** | ~0.7365 | 0.740 | 0.705 |

*(Nota: O valor de 0.764 obtido com o SVM inicial refletiu o melhor desempenho do projeto nos dados privados, garantindo o 4º lugar na classificação geral da competição).*

## 8. Discussão e Conclusões
Através da análise comparativa dos resultados, conclui-se que a agregação dos dados por paciente foi uma estratégia de pré-processamento determinante, prevenindo com eficácia o *data leakage*.

A análise da *Private Leaderboard* revelou dinâmicas de generalização muito interessantes. A **Regressão Logística Otimizada** (C=0.01) demonstrou uma estabilidade assinalável, cravando uma pontuação privada de 0.727 (praticamente idêntica à previsão de 0.7275 da Validação Cruzada). Da mesma forma, o **Random Forest**, ao ser restringido a uma profundidade máxima de 3, evitou memorizar o ruído, alcançando um F1-Score privado de 0.750.

O aspeto mais relevante observou-se no **Support Vector Machine (SVM)**. O modelo inicial alcançou a pontuação máxima do projeto nos dados ocultos (0.764 na *Private Leaderboard*), garantindo o 4º lugar final na competição. A versão que procurámos restringir de forma mais severa (com kernel linear e C=0.01) sofreu uma ligeira quebra de generalização (0.705 no privado). Isto demonstra, na prática, que num conjunto de dados com poucas amostras e alta dimensão de atributos de textura, uma fronteira não-linear moderada provou ser a melhor representação do mundo real.

O uso de pipelines, validação cruzada estratificada e submissões iterativas garantiu não só a reprodutibilidade científica exigida pelo projeto, mas também uma tomada de decisão fundamentada e um resultado altamente competitivo.